In [ ]:
# SUPERINVESTOR DATASET
# import packages:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
import time
import re
import datetime as dt
import math
import yfinance as yf
import numpy as np


def read_table(url):
# A function that extracts tickers and weightings from a portfolio
# displayed on a website specified by the input url

    starting_string = '% change toPortfolio'
    ending_string = 'per page'

    response = requests.get(url)

    if response.status_code == 200:
        bsoup = BeautifulSoup(response.content, 'html.parser')

        get_text = bsoup.get_text()

        starting_index = get_text.find(starting_string)
        ending_index = get_text.find(ending_string, starting_index + len(starting_string))
        
        # Isolate the table with relevant information
        if starting_index != -1 and ending_index != -1:
            extract_text = get_text[starting_index + len(starting_string):ending_index].strip()

        else:
            print(f"Start phrase '{starting_string}' or end phrase '{ending_string}' not found.")
    else:
        print(f"Failed to retrieve webpage. Status code: {response.status_code}")
        
    sections = re.split(r'\n\s*\n', extract_text)

    tickers = []
    weightings = []

    for i in range(len(sections)):
        if "%" in sections[i]:
            continue
        elif "$" in sections[i]:                    # Validate tickers by filtering out characters that 
            continue                                # might appear in the table
        elif " " in sections[i]:
            continue
        elif "," in sections[i]:
            continue
        elif "(" in sections[i]:
            continue
        elif "\n" in sections[i]:
            continue
        elif "-" in sections[i]:
            continue
        elif (len(sections[i]) > 5) & ("." not in sections[i]):
            continue
        elif sections[i] in tickers:
            continue
        elif sections[i] == '1':
            continue
        else:
            tickers.append(sections[i])
            weightings.append(sections[i+2])
            
    return tickers, weightings

In [ ]:
ronald_urls = ['https://valuesider.com/guru/ron-muhlenkamp-co/portfolio?sells_page=1&page=1', 'https://valuesider.com/guru/ron-muhlenkamp-co/portfolio?sells_page=1&page=2']

def construct_weightings(urls):
# This function takes a list of urls and uses read_table to construct the    
# list of tickers and their weightings in the portfolio based on 
# information present in all of the input urls

    full_tickers = []
    full_weightings = []
    
    for url in urls:
        tickers, weightings = read_table(url)
        full_tickers = full_tickers + tickers
        full_weightings = full_weightings + weightings
        
    full_weightings = [item.strip("'").strip("%") for item in full_weightings]
        
    return full_tickers, full_weightings
        

In [ ]:
funds_list = ['FGXXX','FCG','PPA','PDBC','GLD','XAR','GDX','OIH','GDXJ',
             'CATH','SGOV','ACWI','IWF','IWM','BIL','VONG','FXI','KWEB','IWD','SPY','GBTC',
             'SLV','ITB','INDA','EWW','NTPXX','UMBXX','PHYS','GOIXX','VOO','IBIT','EFA',
             'SMIN','KSA','VSS','VEU','VEA','ASHR','IJGXX','FIGXX','ACWX','AMLP']

OTC_list = ['GPLDF','VRAYQ','CABJF','COCXF','HINKF','PHJMF','LRLCF']

# List of all ETFs and Over the Counter stocks that appear in 
# any of the superinvestor portfolios

In [ ]:
def calculate_weighted_dist(tickers, weightings):
# Takes a list of tickers and their corresponding weights
# within the portfolio and returns the distribution of the
# portfolio across business sectors
    
    distribution = []
    technology = 0
    healthcare = 0
    fin_serv = 0
    cons_cyc = 0
    comm_serv = 0
    industrials = 0
    cons_def = 0
    energy = 0
    utilities = 0
    real_estate = 0
    basic_mat = 0
    funds = 0
    
    for i in range(len(tickers)):
        stock = yf.Ticker(str(tickers[i]))
        
        if (tickers[i] in OTC_list) or (tickers[i] == 'LTG'):
            continue
        
        sector = stock.info.get('sector', 'Sector information not available')
        
        if sector == 'Technology':
            technology += float(weightings[i])
        elif sector == 'Healthcare':
            healthcare += float(weightings[i])
        elif sector == 'Financial Services':
            fin_serv += float(weightings[i])
        elif sector == 'Consumer Cyclical':
            cons_cyc += float(weightings[i])
        elif sector == 'Communication Services':
            comm_serv += float(weightings[i])
        elif sector == 'Industrials':
            industrials += float(weightings[i])
        elif sector == 'Consumer Defensive':
            cons_def += float(weightings[i])
        elif sector == 'Energy':
            energy += float(weightings[i])
        elif sector == 'Utilities':
            utilities += float(weightings[i])
        elif sector == 'Real Estate':
            real_estate += float(weightings[i])
        elif sector == 'Basic Materials':
            basic_mat += float(weightings[i])
        else:
            funds += float(weightings[i])
            if tickers[i] not in funds_list:
                print(tickers[i])
                print(sector)
    
    distribution.append(round(technology,2))
    distribution.append(round(healthcare,2))
    distribution.append(round(fin_serv,2))
    distribution.append(round(cons_cyc,2))
    distribution.append(round(comm_serv,2))
    distribution.append(round(industrials,2))
    distribution.append(round(cons_def,2))
    distribution.append(round(energy,2))
    distribution.append(round(utilities,2))
    distribution.append(round(real_estate,2))
    distribution.append(round(basic_mat,2))
    distribution.append(round(funds,2))
    
    return distribution
        

In [ ]:
def validate_tickers(tickers):
# Takes a list of tickers and validates / cleans them
# Some stocks seemed to be listed with different tickers in
# different places. This function manually ensures all stocks
# are identified by a unique ticker
    
    tickers = [ticker[:-3] if ticker.endswith('.WS') else ticker for ticker in tickers]
    
    tickers = [ticker.replace('.', '-') for ticker in tickers]
    
    tickers = [
    'BFH' if ticker == 'ADS' 
    else 'FI' if ticker == 'FISV' 
    else 'CNH' if ticker == 'CNHI'
    else 'CHRD' if ticker == 'OAS'
    else 'VTRS' if ticker == 'VTRSV'
    else 'ELV' if ticker == 'ANTM'
    else 'BALL' if ticker == 'BLL'
    else 'MLKN' if ticker == 'MLHR'
    else 'OKLO' if ticker == 'ALCC'
    else 'GEHC' if ticker == 'GEHCV'
    else 'PARAA' if ticker == 'VIAC'
    else 'NPN.JO' if ticker == 'NPN'
    else '2318.HK' if ticker == '2318'
    else 'WTW' if ticker == 'WLTW'
    else 'MODG' if ticker == 'ELY'
    else 'AZREF' if ticker == 'AZRE'
    else 'GPLDF' if ticker == 'GPL'
    else 'XOM' if ticker == 'PXD'
    else 'FBIN' if ticker == 'FBHS'
    else 'CTRA' if ticker == 'COG'
    else 'NATL' if ticker == 'NCR'
    else 'UPBD' if ticker == 'RCII'
    else 'SBLK' if ticker == 'EGLE'
    else 'RVTY' if ticker == 'PKI'
    else 'VTLE' if ticker == 'LPI'
    else 'DOC' if ticker == 'PEAK'
    else 'IDWM' if ticker == 'IDW'
    else 'NTPIF' if ticker == 'NTP'
    else 'VRAYQ' if ticker == 'VRAY'
    else 'D05.SI' if ticker == 'D05'
    else 'DIDIY' if ticker == 'DIDI'
    else 'PFE' if ticker == 'NBSE'
    else 'AIOT' if ticker == 'MIXT'
    else 'AAPL' if ticker == 'YNDX'
    else 'PFE' if ticker == 'KNTE'
    else ticker
    for ticker in tickers
    ]
    
    return tickers

In [ ]:
brenton_url = ['https://valuesider.com/guru/andrew-brenton-turtle-creek-asset-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/andrew-brenton-turtle-creek-asset-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/andrew-brenton-turtle-creek-asset-management/portfolio?sells_page=1&page=3']

brenton_tickers, brenton_weightings = construct_weightings(brenton_url)

brenton_dist = calculate_weighted_dist(brenton_tickers,brenton_weightings)

print(brenton_dist)
print(len(brenton_tickers))
print((len(brenton_weightings)))
print(sum(brenton_dist))
print('weightings',brenton_weightings)

In [ ]:
super_investors = []
investor_urls = []

super_investors.append('Adam Wyden')
investor_urls.append(['https://valuesider.com/guru/adam-wyden-adw-capital-management/portfolio'])

super_investors.append('Alex Roepers')
investor_urls.append(['https://valuesider.com/guru/alex-roepers-atlantic-investment-management/portfolio'])

super_investors.append('Andrew Brenton')
investor_urls.append(['https://valuesider.com/guru/andrew-brenton-turtle-creek-asset-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/andrew-brenton-turtle-creek-asset-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/andrew-brenton-turtle-creek-asset-management/portfolio?sells_page=1&page=3'])

super_investors.append('Andrew R. Adams')
investor_urls.append(['https://valuesider.com/guru/andrew-adams-mairs-and-power-growth-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/andrew-adams-mairs-and-power-growth-fund/portfolio?sells_page=1&page=2','https://valuesider.com/guru/andrew-adams-mairs-and-power-growth-fund/portfolio?sells_page=1&page=3','https://valuesider.com/guru/andrew-adams-mairs-and-power-growth-fund/portfolio?sells_page=1&page=4'])

super_investors.append('B. Tweedy, Ch. Browne')
investor_urls.append(['https://valuesider.com/guru/bill-tweedy-christopher-browne-tweedy-browne-co/portfolio?sells_page=1&page=1','https://valuesider.com/guru/bill-tweedy-christopher-browne-tweedy-browne-co/portfolio?sells_page=1&page=2','https://valuesider.com/guru/bill-tweedy-christopher-browne-tweedy-browne-co/portfolio?sells_page=1&page=3'])

super_investors.append('Bill Ackman')
investor_urls.append(['https://valuesider.com/guru/bill-ackman-pershing-square-capital-management/portfolio'])

super_investors.append('Bill Gates')
investor_urls.append(['https://valuesider.com/guru/bill-gates-bill--melinda-gates-foundation-trust/portfolio?sells_page=1&page=1','https://valuesider.com/guru/bill-gates-bill--melinda-gates-foundation-trust/portfolio?sells_page=1&page=2'])

super_investors.append('Bill Miller')
investor_urls.append(['https://valuesider.com/guru/bill-miller-miller-value-partners/portfolio?sells_page=1&page=1','https://valuesider.com/guru/bill-miller-miller-value-partners/portfolio?sells_page=1&page=2'])

super_investors.append('Bill Nygren')
investor_urls.append(['https://valuesider.com/guru/bill-nygren-oakmark-select-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/bill-nygren-oakmark-select-fund/portfolio?sells_page=1&page=2'])

super_investors.append('Brian Bares')
investor_urls.append(['https://valuesider.com/guru/bares-capital-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/bares-capital-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/bares-capital-management/portfolio?sells_page=1&page=3'])

super_investors.append('Bruce Berkowitz')
investor_urls.append(['https://valuesider.com/guru/bruce-berkowitz-fairholme-capital-management/portfolio'])

super_investors.append('Bryan R. Lawrence')
investor_urls.append(['https://valuesider.com/guru/bryan-lawrence-oakcliff-capital-partners/portfolio'])

super_investors.append('C.T. Fitzpatrick')
investor_urls.append(['https://valuesider.com/guru/vulcan-value-partners/portfolio?sells_page=1&page=1','https://valuesider.com/guru/vulcan-value-partners/portfolio?sells_page=1&page=2','https://valuesider.com/guru/vulcan-value-partners/portfolio?sells_page=1&page=3','https://valuesider.com/guru/vulcan-value-partners/portfolio?sells_page=1&page=4'])

super_investors.append('Carl Icahn')
investor_urls.append(['https://valuesider.com/guru/carl-icahn-icahn-capital-management/portfolio'])

super_investors.append('Charles Bobrinskoy')
investor_urls.append(['https://valuesider.com/guru/charles-bobrinskoy-ariel-focus-fund/portfolio'])

super_investors.append('Charles Jigarjian')
investor_urls.append(['https://valuesider.com/guru/charles-jigarjian-7g-capital-management/portfolio'])

super_investors.append('Charlie Munger')
investor_urls.append(['https://valuesider.com/guru/charlie-munger-daily-journal-corp/portfolio'])

super_investors.append('Chase Coleman III')
investor_urls.append(['https://valuesider.com/guru/chase-coleman-tiger-global-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/chase-coleman-tiger-global-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/chase-coleman-tiger-global-management/portfolio?sells_page=1&page=3'])

super_investors.append('Chris Hohn')
investor_urls.append(['https://valuesider.com/guru/chris-hohn-tci-fund-management/portfolio'])

super_investors.append('Christopher Bloomstran')
investor_urls.append(['https://valuesider.com/guru/christopher-bloomstran-semper-augustus-investments-group/portfolio?sells_page=1&page=1','https://valuesider.com/guru/christopher-bloomstran-semper-augustus-investments-group/portfolio?sells_page=1&page=2','https://valuesider.com/guru/christopher-bloomstran-semper-augustus-investments-group/portfolio?sells_page=1&page=3'])

super_investors.append('Christopher Davis')
investor_urls.append(['https://valuesider.com/guru/christopher-davis-clipper-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/christopher-davis-clipper-fund/portfolio?sells_page=1&page=2'])

super_investors.append('Chuck Akre')
investor_urls.append(['https://valuesider.com/guru/chuck-akre-akre-capital-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/chuck-akre-akre-capital-management/portfolio?sells_page=1&page=2'])

super_investors.append('Clifford Sosin')
investor_urls.append(['https://valuesider.com/guru/clifford-sosin-cas-investment-partners/portfolio'])

super_investors.append('Connor Haley')
investor_urls.append(['https://valuesider.com/guru/connor-haley-alta-fox-capital-management/portfolio'])

super_investors.append('Daniel Loeb')
investor_urls.append(['https://valuesider.com/guru/daniel-loeb-third-point/portfolio?sells_page=1&page=1','https://valuesider.com/guru/daniel-loeb-third-point/portfolio?sells_page=1&page=2','https://valuesider.com/guru/daniel-loeb-third-point/portfolio?sells_page=1&page=3'])

super_investors.append('David Abrams')
investor_urls.append(['https://valuesider.com/guru/david-abrams-abrams-capital-management/portfolio'])

super_investors.append('David Einhorn')
investor_urls.append(['https://valuesider.com/guru/david-einhorn-greenlight-capital/portfolio?sells_page=1&page=1','https://valuesider.com/guru/david-einhorn-greenlight-capital/portfolio?sells_page=1&page=2','https://valuesider.com/guru/david-einhorn-greenlight-capital/portfolio?sells_page=1&page=3'])

super_investors.append('David Jeffrey Fear')
investor_urls.append(['https://valuesider.com/guru/david-fear-thunderbird-partners/portfolio'])

super_investors.append('David Katz')
investor_urls.append(['https://valuesider.com/guru/david-katz-matrix-advisors-value-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/david-katz-matrix-advisors-value-fund/portfolio?sells_page=1&page=2','https://valuesider.com/guru/david-katz-matrix-advisors-value-fund/portfolio?sells_page=1&page=3'])

super_investors.append('David M. Polen')
investor_urls.append(['https://valuesider.com/guru/david-m-polen-polen-capital-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/david-m-polen-polen-capital-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/david-m-polen-polen-capital-management/portfolio?sells_page=1&page=3','https://valuesider.com/guru/david-m-polen-polen-capital-management/portfolio?sells_page=1&page=4','https://valuesider.com/guru/david-m-polen-polen-capital-management/portfolio?sells_page=1&page=5','https://valuesider.com/guru/david-m-polen-polen-capital-management/portfolio?sells_page=1&page=6','https://valuesider.com/guru/david-m-polen-polen-capital-management/portfolio?sells_page=1&page=7'])

super_investors.append('David Rolfe')
investor_urls.append(['https://valuesider.com/guru/david-rolfe-wedgewood-partners/portfolio?sells_page=1&page=1','https://valuesider.com/guru/david-rolfe-wedgewood-partners/portfolio?sells_page=1&page=2'])

super_investors.append('David Tepper')
investor_urls.append(['https://valuesider.com/guru/david-tepper-appaloosa-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/david-tepper-appaloosa-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/david-tepper-appaloosa-management/portfolio?sells_page=1&page=3'])

super_investors.append('Dennis Hong')
investor_urls.append(['https://valuesider.com/guru/dennis-hong-shawspring-partners/portfolio'])

super_investors.append('Dev Kantesaria')
investor_urls.append(['https://valuesider.com/guru/dev-kantesaria-valley-forge-capital-management/portfolio'])

super_investors.append('Donald G. Smith')
investor_urls.append(['https://valuesider.com/guru/donald-smith-co/portfolio?sells_page=1&page=1','https://valuesider.com/guru/donald-smith-co/portfolio?sells_page=1&page=2','https://valuesider.com/guru/donald-smith-co/portfolio?sells_page=1&page=3','https://valuesider.com/guru/donald-smith-co/portfolio?sells_page=1&page=4','https://valuesider.com/guru/donald-smith-co/portfolio?sells_page=1&page=5'])

super_investors.append('Donald Yacktman')
investor_urls.append(['https://valuesider.com/guru/donald-yacktman-yacktman-asset-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/donald-yacktman-yacktman-asset-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/donald-yacktman-yacktman-asset-management/portfolio?sells_page=1&page=3','https://valuesider.com/guru/donald-yacktman-yacktman-asset-management/portfolio?sells_page=1&page=4','https://valuesider.com/guru/donald-yacktman-yacktman-asset-management/portfolio?sells_page=1&page=5'])

super_investors.append('Duan Yongping')
investor_urls.append(['https://valuesider.com/guru/duan-yongping-h-h-international-investment/portfolio'])

super_investors.append('Edgar Wachenheim III')
investor_urls.append(['https://valuesider.com/guru/edgar-wachenheim-III-greenhaven-associates/portfolio?sells_page=1&page=1','https://valuesider.com/guru/edgar-wachenheim-III-greenhaven-associates/portfolio?sells_page=1&page=2'])

super_investors.append('Eric H. Schoenstein')
investor_urls.append(['https://valuesider.com/guru/eric-schoenstein-jensen-investment-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/eric-schoenstein-jensen-investment-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/eric-schoenstein-jensen-investment-management/portfolio?sells_page=1&page=3','https://valuesider.com/guru/eric-schoenstein-jensen-investment-management/portfolio?sells_page=1&page=4','https://valuesider.com/guru/eric-schoenstein-jensen-investment-management/portfolio?sells_page=1&page=5','https://valuesider.com/guru/eric-schoenstein-jensen-investment-management/portfolio?sells_page=1&page=6'])

super_investors.append('Francis Chou')
investor_urls.append(['https://valuesider.com/guru/francis-chou-chou-associates-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/francis-chou-chou-associates-management/portfolio?sells_page=1&page=2'])

super_investors.append('Francois Rochon')
investor_urls.append(['https://valuesider.com/guru/francois-rochon-giverny-capital/portfolio?sells_page=1&page=1','https://valuesider.com/guru/francois-rochon-giverny-capital/portfolio?sells_page=1&page=2','https://valuesider.com/guru/francois-rochon-giverny-capital/portfolio?sells_page=1&page=3','https://valuesider.com/guru/francois-rochon-giverny-capital/portfolio?sells_page=1&page=4'])

super_investors.append('Fred Martin')
investor_urls.append(['https://valuesider.com/guru/fred-martin-disciplined-growth-investors/portfolio?sells_page=1&page=1','https://valuesider.com/guru/fred-martin-disciplined-growth-investors/portfolio?sells_page=1&page=2','https://valuesider.com/guru/fred-martin-disciplined-growth-investors/portfolio?sells_page=1&page=3','https://valuesider.com/guru/fred-martin-disciplined-growth-investors/portfolio?sells_page=1&page=4','https://valuesider.com/guru/fred-martin-disciplined-growth-investors/portfolio?sells_page=1&page=5'])

super_investors.append('Frederick (Shad) Rowe')
investor_urls.append(['https://valuesider.com/guru/frederick-shad-rowe-greenbrier-partners-capital-management/portfolio'])

super_investors.append('Glenn Greenberg')
investor_urls.append(['https://valuesider.com/guru/glenn-greenberg-brave-warrior-advisors/portfolio?sells_page=1&page=1','https://valuesider.com/guru/glenn-greenberg-brave-warrior-advisors/portfolio?sells_page=1&page=2'])

super_investors.append('Glenn W. Welling')
investor_urls.append(['https://valuesider.com/guru/glenn-welling-engaged-capital/portfolio'])

super_investors.append('Greg Alexander')
investor_urls.append(['https://valuesider.com/guru/greg-alexander-conifer-management/portfolio'])

super_investors.append('Guy Spier')
investor_urls.append(['https://valuesider.com/guru/guy-spier-aquamarine-capital/portfolio'])

super_investors.append('Harry Burn')
investor_urls.append(['https://valuesider.com/guru/harry-burn-sound-shore-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/harry-burn-sound-shore-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/harry-burn-sound-shore-management/portfolio?sells_page=1&page=3'])

super_investors.append('Henry Ellenbogen')
investor_urls.append(['https://valuesider.com/guru/henry-ellenbogen-durable-capital-partners/portfolio?sells_page=1&page=1','https://valuesider.com/guru/henry-ellenbogen-durable-capital-partners/portfolio?sells_page=1&page=2','https://valuesider.com/guru/henry-ellenbogen-durable-capital-partners/portfolio?sells_page=1&page=3','https://valuesider.com/guru/henry-ellenbogen-durable-capital-partners/portfolio?sells_page=1&page=4'])

super_investors.append('Howard Marks')
investor_urls.append(['https://valuesider.com/guru/howard-marks-oaktree-capital-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/howard-marks-oaktree-capital-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/howard-marks-oaktree-capital-management/portfolio?sells_page=1&page=3','https://valuesider.com/guru/howard-marks-oaktree-capital-management/portfolio?sells_page=1&page=4'])

super_investors.append('Independent Franchise Partners')
investor_urls.append(['https://valuesider.com/guru/independent-franchise-partners-us-equity-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/independent-franchise-partners-us-equity-fund/portfolio?sells_page=1&page=2'])

super_investors.append('Jeffrey Ubben')
investor_urls.append(['https://valuesider.com/guru/jeffrey-ubben-valueact-holdings/portfolio'])

super_investors.append('Jim Cullen')
investor_urls.append(['https://valuesider.com/guru/jim-cullen-capital-management-value-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/jim-cullen-capital-management-value-fund/portfolio?sells_page=1&page=2','https://valuesider.com/guru/jim-cullen-capital-management-value-fund/portfolio?sells_page=1&page=3'])

super_investors.append('John Armitage')
investor_urls.append(['https://valuesider.com/guru/john-armitage-egerton-capital/portfolio?sells_page=1&page=1','https://valuesider.com/guru/john-armitage-egerton-capital/portfolio?sells_page=1&page=2'])

super_investors.append('John W. Rogers Jr.')
investor_urls.append(['https://valuesider.com/guru/john-rogers-ariel-appreciation-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/john-rogers-ariel-appreciation-fund/portfolio?sells_page=1&page=2','https://valuesider.com/guru/john-rogers-ariel-appreciation-fund/portfolio?sells_page=1&page=3'])

super_investors.append('Josh Tarasoff')
investor_urls.append(['https://valuesider.com/guru/josh-tarasoff-greenlea-lane-capital-management/portfolio'])

super_investors.append('Li Lu')
investor_urls.append(['https://valuesider.com/guru/li-lu-himalaya-capital-management/portfolio'])

super_investors.append('Mark A. Hillman')
investor_urls.append(['https://valuesider.com/guru/mark-hillman-value-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/mark-hillman-value-fund/portfolio?sells_page=1&page=2','https://valuesider.com/guru/mark-hillman-value-fund/portfolio?sells_page=1&page=3'])

super_investors.append('Mark Massey')
investor_urls.append(['https://valuesider.com/guru/mark-massey-altarock-partners/portfolio'])

super_investors.append('Marty Whitman')
investor_urls.append(['https://valuesider.com/guru/marty-whitman-third-avenue-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/marty-whitman-third-avenue-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/marty-whitman-third-avenue-management/portfolio?sells_page=1&page=3','https://valuesider.com/guru/marty-whitman-third-avenue-management/portfolio?sells_page=1&page=4'])

super_investors.append('Mason Hawkins')
investor_urls.append(['https://valuesider.com/guru/mason-hawkins-longleaf-partners/portfolio?sells_page=1&page=1','https://valuesider.com/guru/mason-hawkins-longleaf-partners/portfolio?sells_page=1&page=2'])

super_investors.append('Michael Burry')
investor_urls.append(['https://valuesider.com/guru/michael-burry-scion-asset-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/michael-burry-scion-asset-management/portfolio?sells_page=1&page=2'])

super_investors.append('Michael Lindsell')
investor_urls.append(['https://valuesider.com/guru/michael-lindsell-nick-train-lindsell-train/portfolio?sells_page=1&page=1','https://valuesider.com/guru/michael-lindsell-nick-train-lindsell-train/portfolio?sells_page=1&page=2'])

super_investors.append('Mohnish Pabrai')
investor_urls.append(['https://valuesider.com/guru/mohnish-pabrai-dalal-street/portfolio'])

super_investors.append('Nathaniel Simons')
investor_urls.append(['https://valuesider.com/guru/nat-simons-meritage-group/portfolio?sells_page=1&page=1','https://valuesider.com/guru/nat-simons-meritage-group/portfolio?sells_page=1&page=2','https://valuesider.com/guru/nat-simons-meritage-group/portfolio?sells_page=1&page=3','https://valuesider.com/guru/nat-simons-meritage-group/portfolio?sells_page=1&page=4'])

super_investors.append('Nelson Peltz')
investor_urls.append(['https://valuesider.com/guru/nelson-peltz-trian-fund-management/portfolio'])

super_investors.append('Nicolai Tangen')
investor_urls.append(['https://valuesider.com/guru/nicolai-tangen-ako-capital/portfolio?sells_page=1&page=1','https://valuesider.com/guru/nicolai-tangen-ako-capital/portfolio?sells_page=1&page=2'])

super_investors.append('Norbert Lou')
investor_urls.append(['https://valuesider.com/guru/norbert-lou-punch-card-management/portfolio'])

super_investors.append('Ole Andreas Halvorsen')
investor_urls.append(['https://valuesider.com/guru/andreas-halvorsen-viking-global-investors/portfolio?sells_page=1&page=1','https://valuesider.com/guru/andreas-halvorsen-viking-global-investors/portfolio?sells_page=1&page=2','https://valuesider.com/guru/andreas-halvorsen-viking-global-investors/portfolio?sells_page=1&page=3','https://valuesider.com/guru/andreas-halvorsen-viking-global-investors/portfolio?sells_page=1&page=4','https://valuesider.com/guru/andreas-halvorsen-viking-global-investors/portfolio?sells_page=1&page=5','https://valuesider.com/guru/andreas-halvorsen-viking-global-investors/portfolio?sells_page=1&page=6'])

super_investors.append('Parnassus Investments')
investor_urls.append(['https://valuesider.com/guru/parnassus-endeavor-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/parnassus-endeavor-fund/portfolio?sells_page=1&page=2','https://valuesider.com/guru/parnassus-endeavor-fund/portfolio?sells_page=1&page=3','https://valuesider.com/guru/parnassus-endeavor-fund/portfolio?sells_page=1&page=4'])

super_investors.append('Pat Dorsey')
investor_urls.append(['https://valuesider.com/guru/pat-dorsey-dorsey-asset-management/portfolio'])

super_investors.append('Paul Isaac')
investor_urls.append(['https://valuesider.com/guru/paul-isaac-arbiter-partners-capital-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/paul-isaac-arbiter-partners-capital-management/portfolio?sells_page=1&page=2'])

super_investors.append('Paul Lountzis')
investor_urls.append(['https://valuesider.com/guru/paul-lountzis-asset-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/paul-lountzis-asset-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/paul-lountzis-asset-management/portfolio?sells_page=1&page=3','https://valuesider.com/guru/paul-lountzis-asset-management/portfolio?sells_page=1&page=4'])

super_investors.append('Phil Town')
investor_urls.append(['https://valuesider.com/guru/phil-town-rule-one-fund/portfolio'])

super_investors.append('Prem Watsa')
investor_urls.append(['https://valuesider.com/guru/prem-watsa-fairfax-financial-holdings/portfolio?sells_page=1&page=1','https://valuesider.com/guru/prem-watsa-fairfax-financial-holdings/portfolio?sells_page=1&page=2','https://valuesider.com/guru/prem-watsa-fairfax-financial-holdings/portfolio?sells_page=1&page=3'])

super_investors.append('Quincy Lee')
investor_urls.append(['https://valuesider.com/guru/quincy-lee-ancient-art-teton-capital/portfolio?sells_page=1&page=1','https://valuesider.com/guru/quincy-lee-ancient-art-teton-capital/portfolio?sells_page=1&page=2'])

super_investors.append('Ravenel Boykin Curry IV')
investor_urls.append(['https://valuesider.com/guru/ravenel-boykin-curry-eagle-capital-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/ravenel-boykin-curry-eagle-capital-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/ravenel-boykin-curry-eagle-capital-management/portfolio?sells_page=1&page=3','https://valuesider.com/guru/ravenel-boykin-curry-eagle-capital-management/portfolio?sells_page=1&page=4'])

super_investors.append('Rob Vinall')
investor_urls.append(['https://valuesider.com/guru/rob-vinall-rv-capital/portfolio'])

super_investors.append('Robert Karr')
investor_urls.append(['https://valuesider.com/guru/robert-karr-joho-capital/portfolio?sells_page=1&page=1','https://valuesider.com/guru/robert-karr-joho-capital/portfolio?sells_page=1&page=2'])

super_investors.append('Robert Olstein')
investor_urls.append(['https://valuesider.com/guru/robert-olstein-olstein-capital-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/robert-olstein-olstein-capital-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/robert-olstein-olstein-capital-management/portfolio?sells_page=1&page=3','https://valuesider.com/guru/robert-olstein-olstein-capital-management/portfolio?sells_page=1&page=4','https://valuesider.com/guru/robert-olstein-olstein-capital-management/portfolio?sells_page=1&page=5','https://valuesider.com/guru/robert-olstein-olstein-capital-management/portfolio?sells_page=1&page=6','https://valuesider.com/guru/robert-olstein-olstein-capital-management/portfolio?sells_page=1&page=7'])

super_investors.append('Robert Torray')
investor_urls.append(['https://valuesider.com/guru/robert-torray-torray-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/robert-torray-torray-fund/portfolio?sells_page=1&page=2'])

super_investors.append('Ronald Muhlenkamp')
investor_urls.append(['https://valuesider.com/guru/ron-muhlenkamp-co/portfolio?sells_page=1&page=1','https://valuesider.com/guru/ron-muhlenkamp-co/portfolio?sells_page=1&page=2'])

super_investors.append('Ruane, Cunnif & Goldfarb')
investor_urls.append(['https://valuesider.com/guru/ruane-cunniff--goldfarb-sequoia-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/ruane-cunniff--goldfarb-sequoia-fund/portfolio?sells_page=1&page=2'])

super_investors.append('Sarah Ketterer')
investor_urls.append(['https://valuesider.com/guru/sarah-ketterer-causeway-capital-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/sarah-ketterer-causeway-capital-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/sarah-ketterer-causeway-capital-management/portfolio?sells_page=1&page=3','https://valuesider.com/guru/sarah-ketterer-causeway-capital-management/portfolio?sells_page=1&page=4','https://valuesider.com/guru/sarah-ketterer-causeway-capital-management/portfolio?sells_page=1&page=5','https://valuesider.com/guru/sarah-ketterer-causeway-capital-management/portfolio?sells_page=1&page=6'])

super_investors.append('Seth Klarman')
investor_urls.append(['https://valuesider.com/guru/seth-klarman-baupost-group/portfolio?sells_page=1&page=1','https://valuesider.com/guru/seth-klarman-baupost-group/portfolio?sells_page=1&page=2'])

super_investors.append('Stephen Mandel')
investor_urls.append(['https://valuesider.com/guru/stephen-mandel-lone-pine-capital/portfolio?sells_page=1&page=1','https://valuesider.com/guru/stephen-mandel-lone-pine-capital/portfolio?sells_page=1&page=2'])

super_investors.append('Stuart Mclaughlin')
investor_urls.append(['https://valuesider.com/guru/triple-frond-partners/portfolio'])

super_investors.append('Terry Smith')
investor_urls.append(['https://valuesider.com/guru/terry-smith-fundsmith/portfolio?sells_page=1&page=1','https://valuesider.com/guru/terry-smith-fundsmith/portfolio?sells_page=1&page=2','https://valuesider.com/guru/terry-smith-fundsmith/portfolio?sells_page=1&page=3'])

super_investors.append('Thomas Graham, Alan, Irving Kahns')
investor_urls.append(['https://valuesider.com/guru/thomas-graham-alan-irving-kahns-kahn-brothers-group/portfolio?sells_page=1&page=1','https://valuesider.com/guru/thomas-graham-alan-irving-kahns-kahn-brothers-group/portfolio?sells_page=1&page=2','https://valuesider.com/guru/thomas-graham-alan-irving-kahns-kahn-brothers-group/portfolio?sells_page=1&page=3','https://valuesider.com/guru/thomas-graham-alan-irving-kahns-kahn-brothers-group/portfolio?sells_page=1&page=4'])

super_investors.append('Thomas Russo')
investor_urls.append(['https://valuesider.com/guru/thomas-russo-gardner-russo--quinn/portfolio?sells_page=1&page=1','https://valuesider.com/guru/thomas-russo-gardner-russo--quinn/portfolio?sells_page=1&page=2','https://valuesider.com/guru/thomas-russo-gardner-russo--quinn/portfolio?sells_page=1&page=3','https://valuesider.com/guru/thomas-russo-gardner-russo--quinn/portfolio?sells_page=1&page=4','https://valuesider.com/guru/thomas-russo-gardner-russo--quinn/portfolio?sells_page=1&page=5','https://valuesider.com/guru/thomas-russo-gardner-russo--quinn/portfolio?sells_page=1&page=6'])

super_investors.append('V.D. Dodge, E. M. Cox')
investor_urls.append(['https://valuesider.com/guru/van-duyn-dodge-e-morris-cox-dodge--cox/portfolio?sells_page=1&page=1','https://valuesider.com/guru/van-duyn-dodge-e-morris-cox-dodge--cox/portfolio?sells_page=1&page=2','https://valuesider.com/guru/van-duyn-dodge-e-morris-cox-dodge--cox/portfolio?sells_page=1&page=3','https://valuesider.com/guru/van-duyn-dodge-e-morris-cox-dodge--cox/portfolio?sells_page=1&page=4','https://valuesider.com/guru/van-duyn-dodge-e-morris-cox-dodge--cox/portfolio?sells_page=1&page=5','https://valuesider.com/guru/van-duyn-dodge-e-morris-cox-dodge--cox/portfolio?sells_page=1&page=6'])

super_investors.append('Wallace Weitz')
investor_urls.append(['https://valuesider.com/guru/wallace-weitz-value-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/wallace-weitz-value-fund/portfolio?sells_page=1&page=2','https://valuesider.com/guru/wallace-weitz-value-fund/portfolio?sells_page=1&page=3'])

super_investors.append('Warren Buffet')
investor_urls.append(['https://valuesider.com/guru/warren-buffett-berkshire-hathaway/portfolio?sells_page=1&page=1','https://valuesider.com/guru/warren-buffett-berkshire-hathaway/portfolio?sells_page=1&page=2','https://valuesider.com/guru/warren-buffett-berkshire-hathaway/portfolio?sells_page=1&page=3'])

super_investors.append('William Von Mueffling')
investor_urls.append(['https://valuesider.com/guru/william-von-mueffling-cantillon-capital-management/portfolio?sells_page=1&page=1','https://valuesider.com/guru/william-von-mueffling-cantillon-capital-management/portfolio?sells_page=1&page=2','https://valuesider.com/guru/william-von-mueffling-cantillon-capital-management/portfolio?sells_page=1&page=3'])

In [ ]:
technology_col = []
healthcare_col = []
fin_serv_col = []
cons_cyc_col = []
comm_serv_col = []
industrials_col = []
cons_def_col = []
energy_col = []
utilities_col = []
real_estate_col = []
basic_mat_col = []
funds_col = []
sums = []

# Construct sector distribution for all urls associated with
# each of the superinvestors

for url_list in investor_urls:
    
    print(url_list[0])
    
    investor_tickers,investor_weightings = construct_weightings(url_list)
    
    investor_tickers = validate_tickers(investor_tickers)
    
    investor_dist = calculate_weighted_dist(investor_tickers,investor_weightings)
    
    technology_col.append(investor_dist[0])
    healthcare_col.append(investor_dist[1])
    fin_serv_col.append(investor_dist[2])
    cons_cyc_col.append(investor_dist[3])
    comm_serv_col.append(investor_dist[4])
    industrials_col.append(investor_dist[5])
    cons_def_col.append(investor_dist[6])
    energy_col.append(investor_dist[7])
    utilities_col.append(investor_dist[8])
    real_estate_col.append(investor_dist[9])
    basic_mat_col.append(investor_dist[10])
    funds_col.append(investor_dist[11])
    sums.append(round(sum(investor_dist),2))

In [ ]:
check_ticker,check_weightings = construct_weightings(['https://valuesider.com/guru/christopher-davis-clipper-fund/portfolio?sells_page=1&page=1','https://valuesider.com/guru/christopher-davis-clipper-fund/portfolio?sells_page=1&page=2'])

In [ ]:
print(len(check_ticker))
print(len(check_weightings))
print(check_ticker)
print(check_weightings)

In [ ]:
len(check_ticker[11])

In [ ]:
print(sums)
print(len(sums))

In [ ]:
data_table = {
    'Superinvestor': super_investors,
    'Technology': technology_col,
    'Healthcare': healthcare_col,                  # Dataframe with all superinvestor portfolio distributions
    'Financial Services': fin_serv_col,
    'Consumer Cyclical': cons_cyc_col,
    'Communication Services': comm_serv_col,
    'Industrials': industrials_col,
    'Consumer Defensive': cons_def_col,
    'Energy': energy_col,
    'Utilities': utilities_col,
    'Real Estate': real_estate_col,
    'Basic Materials': basic_mat_col,
    'Funds': funds_col
}

superinvestor_data = pd.DataFrame(data_table)
superinvestor_data.to_csv('../data/superinvestor-data.csv')

In [ ]:
# CALCULATE PREVIOUS YEAR PRICE CHANGE

tickers_record = []
weightings_record = []

for url_list in investor_urls:
    
    print(url_list[0])
    
    investor_tickers,investor_weightings = construct_weightings(url_list)
    
    investor_tickers = validate_tickers(investor_tickers)
    
    tickers_record.append(investor_tickers)
    weightings_record.append(investor_weightings)

In [ ]:
# CALCULATE PREVIOUS YEAR PRICE CHANGE

earnings = []

for i in range(len(super_investors)):
    
    start_price = 0
    end_price = 0
    local_tickers = tickers_record[i]
    local_weightings = weightings_record[i]
    
    for j in range(len(local_tickers)):
        
        stock_info = yf.download(local_tickers[j],'2023-07-01','2024-07-01')
        print(float(stock_info['Open'][0]))
        print(float(local_weightings[j]))
        ticker_open = float(stock_info['Open'][0])
        ticker_close = float(stock_info['Close'][-1])
        ticker_weight = float(local_weightings[j]) / 100
        
        start_price += ticker_open * ticker_weight
        end_price += ticker_close * ticker_weight
        
    earnings_i = (float(end_price) - float(start_price)) / float(start_price) * 100
    earnings.append(earnings_i)
    
    
    
    